<a id='9c-11'></a>

## 9c-11. 🧩 Pattern 11: Callables & First-Class Functions — LC 912, 215

---

```
PROBLEM:  Functions are objects. Pass them, store them, return them, call them later.

APPROACH: Treat functions like any other value — assign to variables, put in dicts,
          pass as arguments. Combine with __call__ to make objects callable too.

SLOW MOTION TRACE:
  # Functions as values
  ops = {'+': lambda a,b: a+b, '-': lambda a,b: a-b}
  step 1   ops['+']        -> <function>
  step 2   ops['+'](3, 4)  -> 7          (called by key lookup)

  # Returning a function — factory pattern
  def make_multiplier(n):
      return lambda x: x * n     # n is "baked in" via closure
  triple = make_multiplier(3)
  step 1   triple(5)  -> 15      (n=3 was captured in closure)

  # __call__ — make any object callable
  class Adder:
      def __init__(self, n): self.n = n
      def __call__(self, x): return x + self.n
  add5 = Adder(5)
  step 1   add5(10)  -> 15       (instance called like a function)

KEY INSIGHT: When anything can be called the same way, you can swap algorithms
             without changing the calling code — that's the strategy pattern.

TIME / SPACE: No complexity overhead — callable dispatch is O(1) dictionary lookup.
```


In [ ]:
from typing import List, Callable, Dict


# ── Functions as first-class values ──────────────────────────────────────────
# Assign a function to a variable — same object, new name
def double(x): return x * 2
def triple(x): return x * 3

transform = double          # no () — we're storing, not calling
print(transform(5))         # 10  — now we call it

# Store functions in a dict — O(1) dispatch without if/elif chains
ops: Dict[str, Callable[[int, int], int]] = {
    '+': lambda a, b: a + b,
    '-': lambda a, b: a - b,
    '*': lambda a, b: a * b,
}
print(ops['+'](3, 4))       # 7
print(ops['*'](3, 4))       # 12


# ── Passing functions as arguments ───────────────────────────────────────────
def apply_to_each(nums: List[int], fn: Callable[[int], int]) -> List[int]:
    """Apply fn to every element — caller decides the transformation."""
    return [fn(x) for x in nums]

print(apply_to_each([1, 2, 3], double))    # [2, 4, 6]
print(apply_to_each([1, 2, 3], triple))    # [3, 6, 9]
print(apply_to_each([1, 2, 3], lambda x: x ** 2))  # [1, 4, 9]


# ── Factory: returning a function (closure) ───────────────────────────────────
# Think: a vending machine that dispenses custom multipliers
def make_multiplier(n: int) -> Callable[[int], int]:
    """
    Factory pattern — bake n into the returned function via closure.
    The returned lambda 'remembers' n even after make_multiplier returns.
    """
    return lambda x: x * n    # n lives in the closure, not in the lambda signature

times_5  = make_multiplier(5)
times_10 = make_multiplier(10)
print(times_5(7))             # 35
print(times_10(7))            # 70

# LC 215 — Kth Largest: strategy swap demo
def find_kth_largest_sort(nums: List[int], k: int) -> int:
    """
    LC 215 — Kth Largest Element using sort strategy.
    Approach: sort descending, index k-1.
    Args:
        nums (List[int]): unsorted list.
        k (int): rank from top.
    Returns:
        int: kth largest value.
    Time:  O(n log n) — full sort
    Space: O(1) — in-place sort
    """
    nums.sort(reverse=True)     # sort descending — kth largest is at index k-1
    return nums[k - 1]

def test_harness_kth(fn):
    tests = [
        ([3, 2, 1, 5, 6, 4], 2, 5),
        ([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4),
        ([1], 1, 1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_kth(find_kth_largest_sort)
print("find_kth_largest_sort defined.")


# ── __call__: making objects callable ────────────────────────────────────────
# Think: a robot with a button — pressing it runs its main job
class Adder:
    """Callable object that adds a fixed offset."""
    def __init__(self, n: int):
        self.n = n
    def __call__(self, x: int) -> int:
        return x + self.n             # called when you do instance(x)

add5  = Adder(5)
add10 = Adder(10)
print(add5(3))      # 8    — instance called like a function
print(add10(3))     # 13
print(callable(add5))   # True  — __call__ makes it callable

# Simplicity and clarity is Gold
